## Imports and helper functions

In [ ]:
import pprint
import struct
from array import array
from collections.abc import Iterable
from dataclasses import dataclass, field
from enum import IntEnum
from pathlib import Path
from typing import BinaryIO

# Please install PIL or Pillow:
from PIL.ImagePalette import ImagePalette
from PIL import Image, ImageSequence

from IPython.display import display

In [ ]:
# Why isn't this a built-in function already?
def unpack_stream(format: str, stream: BinaryIO) -> tuple:
    size = struct.calcsize(format)
    buf = stream.read(size)
    assert len(buf) == size
    assert isinstance(buf, bytes)
    return struct.unpack(format, buf)

## Jetpack classes

In [ ]:
class JetpackEnemyKind(IntEnum):
    NONE = 0
    TRACKBOT = 1
    STEEL_BALL = 2
    SPRING = 3
    MISSILE = 4
    SPIKES = 5
    FLITZER = 6
    HOMER = 7
    BATBOT = 8

In [ ]:
@dataclass
class JetpackEnemy:
    kind: JetpackEnemyKind = JetpackEnemyKind.NONE
    x: int = 0
    y: int = 0

    @classmethod
    def unpack(cls, stream: BinaryIO) -> JetpackEnemy:
        obj = cls()
        obj.kind = JetpackEnemyKind(*unpack_stream('B', f))
        obj.x, obj.y = unpack_stream('BB', f)
        return obj

In [ ]:
@dataclass
class JetpackTilemap:
    # This internal data could have been implemented in several ways:
    # - bytes (immutable)
    # - bytearray (mutable array of bytes)
    # - array (mutable array of 8-bit integers or other word sizes)
    data: bytearray = field(default_factory=lambda: bytearray(16 * 26))
    width: int = 26
    height: int = 16

    @classmethod
    def unpack(cls, stream: BinaryIO, *, width=None, height=None) -> JetpackTilemap:
        obj = cls()
        if width is not None:
            obj.width = width
        if height is not None:
            obj.height = height
        (data,) = unpack_stream('{}s'.format(len(obj)), f)
        obj.data = bytearray(data)
        return obj

    def __len__(self) -> int:
        return self.width * self.height

    def __getitem__(self, subscript) -> int:
        '''Supports multiple syntaxes:
        tilemap[123] -> int in range 0..len(tilemap)
        tilemap[12:34] -> slice of two (or three) integers
        tilemap[4, 6] -> separate coordinates for x and y
        '''
        if isinstance(subscript, (int, slice)):
            return self.data[subscript]
        elif isinstance(subscript, tuple):
            x, y = subscript
            if 0 <= x < self.width and 0 <= y < self.height:
                return self.data[x + y * self.width]
            else:
                raise ValueError('Out-of-bounds x or y coordinates: {}, {}'.format(x, y))
        else:
            raise TypeError('Expected int or tuple, got {!r}'.format(subscript))

    def items(self) -> Iterable[tuple]:
        '''Generator of tuples (x, y, tile), for each of the tiles in this tilemap.
        '''
        for y in range(self.height):
            for x in range(self.width):
                yield (x, y, self[x, y])

    def rows(self) -> Iterable[bytes]:
        for y in range(self.height):
            yield self.data[y * self.width : (y+1) * self.width]

In [ ]:
@dataclass
class JetpackLevel:
    #tiles: array|bytes|bytearray = field(default_factory=lambda: bytearray(16 * 26))
    tilemap: JetpackTilemap = field(default_factory=JetpackTilemap)
    door_x: int = 0 
    door_y: int = 0
    player_x: int = 0
    player_y: int = 0
    enemies: list[JetpackEnemy] = field(default_factory=lambda: [JetpackEnemy() for _ in range(20)])
    description: bytes|bytearray = b' ' * 26
    filename: str = ''

    @classmethod
    def unpack(cls, stream: BinaryIO, filename: str|Path = '') -> JetpackLevel:
        obj = cls()
        obj.filename = filename
        #(obj.tiles,) = unpack_stream('416s', f)
        obj.tilemap = JetpackTilemap.unpack(f)
        obj.door_x, obj.door_y = unpack_stream('BB', f)
        obj.player_x, obj.player_y = unpack_stream('BB', f)
        obj.enemies = [JetpackEnemy.unpack(f) for _ in range(20)]
        (obj.description,) = unpack_stream('26s', f)
        return obj

## Tilemap classes (and functions) (i.e. gfx-related)

In [ ]:
class JetpackEditorTileset:
    def __init__(self, filename):
        self.filename = filename
        self.image = Image.open(self.filename)

    def __repr__(self) -> str:
        return 'JetpackEditorTileset({!r})'.format(self.filename)

    def _get_tile_box(self, id: int) -> tuple[int]:
        if not 0 <= id < 120:
            raise ValueError('Invalid tile {!r}'.format(id))

        tiles_per_row = 20
        x = id % tiles_per_row
        y = id // tiles_per_row
        w = h = 12
        dx = dy = 16

        px = 2 + x * dx
        py = 96 + y * dy
        return (px, py, px + w, py + h)

    def get_tile(self, id: int) -> Image.Image:
        box = self._get_tile_box(id)
        return self.image.crop(box)

    def get_tiles(self, id: int) -> list[Image.Image]:
        box = self._get_tile_box(id)
        return ImageSequence.all_frames(self.image, lambda image: image.crop(box))

In [ ]:
def image_from_level(level: JetpackLevel, tileset: JetpackEditorTileset) -> Image.Image:
    w = h = 12
    output = Image.new(tileset.image.mode, (w * level.tilemap.width, h * level.tilemap.height))
    for (x, y, tile) in level.tilemap.items():
        output.paste(tileset.get_tile(tile), (x * w, y * h))

    # TODO:
    # - Animation frames
    # - Add enemies
    # - Add door and player
    # - Separate layers
    # - Proper background/foreground, including transparency
    return output

## Exploration

In [ ]:
foo = JetpackLevel()
pprint.pprint(foo)

In [ ]:
pathname = Path('./levelpacks/xmasjetpack/XMAS0.JET')
with open(pathname, 'rb') as f:
    foo = JetpackLevel.unpack(f, pathname.name)
    pprint.pprint(foo)

In [ ]:
foo.tilemap[32-26, 1]

In [ ]:
foo.tilemap[0:26]

In [ ]:
try:
    foo.tilemap[0:5,2:26]
except TypeError as e:
    print(e)

In [ ]:
list(foo.tilemap.rows())

In [ ]:
list(foo.tilemap.items())

In [ ]:
with Image.open('./Editor-tileset.webp') as tileset:
    print(tileset.n_frames)

In [ ]:
tileset

In [ ]:

tileset.crop((2, 96, 2+12, 96+12))

In [ ]:
tileset = JetpackEditorTileset('../Editor-tileset.webp')
tileset

In [ ]:
for tile in range(0, 120):
    display(tileset.get_tile(tile))

In [ ]:
for im in tileset.get_tiles(2):
    display(im)

In [ ]:
image_from_level(foo, tileset)